<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>HW2 @ Deep Learning Course, Dr. Soleymani</b></p>
    <p><b>Desinged by Shaygan Adim</b></p>
</div>

---



## Homework Overview
In this homework, you will learn to implement, train, and evaluate Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) models on a text classification task using a dataset of IMDB movie reviews, and compare them.

**NOTE : Be sure to answer the analytical questions at the end of the notebook as well.**

In [2]:
# Imports
import nltk
nltk.download('stopwords')
import os
import random
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk import wordpunct_tokenize
from tqdm import tqdm
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from IPython.core.display import display, HTML
from datasets import load_dataset

# Enable tqdm progress bar in pandas
tqdm.pandas()

# Set device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

[nltk_data] Downloading package stopwords to
[nltk_data]     ~\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
~\AppData\Local\Temp\ipykernel_12772\4217334980.py:20: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


Using device: cpu


# Dataset

In this section, we’ll load the IMDB dataset and preprocess the data to make it suitable for training RNN and LSTM models.

## Load Dataset
Description of Dataset: The IMDB movie reviews dataset consists of reviews along with their labels (positive or negative sentiment). Each review is a sentence or paragraph of text.

Download the Dataset: We will use a Huggingface to download the dataset into our environment.

In [3]:
# Load the IMDb dataset (Hugging Face)
dataset = load_dataset("imdb")

# Make sure 'data' folder exists
os.makedirs('data', exist_ok=True)

# Combine train and test splits into one DataFrame
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])
full_df = pd.concat([train_df, test_df], ignore_index=True)

# Save combined DataFrame to CSV
DATA_PATH = 'data/imdb_reviews.csv'
full_df.to_csv(DATA_PATH, index=False)

print(f"✅ Saved combined IMDb reviews to: {DATA_PATH}")
print(f"Total samples: {len(full_df)}")

README.md: 0.00B [00:00, ?B/s]

~\anaconda3\envs\dl_hw1\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in ~\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

✅ Saved combined IMDb reviews to: data/imdb_reviews.csv
Total samples: 50000


In [4]:
# Show 5 random samples from the DataFrame
print(full_df.sample(5))

                                                    text  label
29861  AVP2 is an awful movie. The dialogue was point...      0
26133  Rented this one by mistake thinking it was ano...      0
14564  First, there is NO way the remake can be as go...      1
47528  I am really sad that that this film has got so...      1
41541  Brown returned to his role from the year befor...      1


## Preprocessing

For our models to work effectively, we need to preprocess the text data by cleaning it and converting words to integer indices for training.Preproces steps
such as Tokenization and Cleaning , Replacing Rare Words , Build Vocabulary , Convert Tokens to Indices and Prepare Data for Training.

**NOTE : Do not alter the structure of this preprocessing code, as it aligns with other parts of the notebook.However, minor adjustments for compatibility with your code are allowed if needed.**

In [5]:
def tokenize(text, stop_words):
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    tokens = wordpunct_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

In [6]:
def remove_rare_words(tokens, common_tokens, max_len):
    return [token if token in common_tokens
            else '<UNK>' for token in tokens][-max_len:]

In [7]:
def load_and_preprocess_data(data_path, max_vocab, max_len):
    df = pd.read_csv(data_path)
    stop_words = set(stopwords.words('english'))

    # Clean and tokenize
    df['tokens'] = df['text'].apply(lambda x: tokenize(x, stop_words))

    # Replace rare words with <UNK>
    all_tokens = [token for tokens in df['tokens'] for token in tokens]
    common_tokens = set(list(zip(*Counter(all_tokens).most_common(max_vocab)))[0])
    df['tokens'] = df['tokens'].apply(lambda x: remove_rare_words(x, common_tokens, max_len))

    # Remove sequences with only <UNK>
    df = df[df['tokens'].apply(lambda tokens: any(token != '<UNK>' for token in tokens))]

    # Build vocab
    vocab = sorted(set([token for tokens in df['tokens'] for token in tokens]))
    token2idx = {token: idx for idx, token in enumerate(vocab)}
    token2idx['<PAD>'] = len(token2idx)

    # Index tokens
    df['indexed_tokens'] = df['tokens'].apply(lambda tokens: [token2idx[token] for token in tokens])

    return df['indexed_tokens'].tolist(), df['label'].tolist(), token2idx

In [8]:
# How many of the most common vocab words to keep
# Uncommon words get replaced with unknown token <UNK>
max_vocab = 2500

# How many tokens long each sequence will be cut to
# Shorter sequences will get the padding token <PAD>
max_len = 100

sequences, targets, token2idx = load_and_preprocess_data(DATA_PATH, max_vocab, max_len)


In [9]:
def split_data(sequences, targets, valid_ratio=0.05, test_ratio=0.05):
    total_size = len(sequences)
    test_size = int(total_size * test_ratio)
    valid_size = int(total_size * valid_ratio)
    train_size = total_size - valid_size - test_size

    train_sequences, train_targets = sequences[:train_size], targets[:train_size]
    valid_sequences, valid_targets = sequences[train_size:train_size + valid_size], targets[train_size:train_size + valid_size]
    test_sequences, test_targets = sequences[train_size + valid_size:], targets[train_size + valid_size:]

    return train_sequences, train_targets, valid_sequences, valid_targets, test_sequences, test_targets

In [10]:
train_sequences, train_targets, valid_sequences, valid_targets, test_sequences, test_targets = split_data(sequences, targets)

In [11]:
def collate(batch):
    inputs, targets = zip(*batch)
    inputs_padded = pad_sequences(inputs, padding_val=token2idx['<PAD>'])
    return torch.LongTensor(inputs_padded), torch.LongTensor(targets)

In [12]:
def pad_sequences(sequences, padding_val=0, pad_left=False):
    """Pad a list of sequences to the same length with a padding_val."""
    sequence_length = max(len(sequence) for sequence in sequences)
    if not pad_left:
        return [sequence + [padding_val] * (sequence_length - len(sequence)) for sequence in sequences]
    return [[padding_val] * (sequence_length - len(sequence)) + sequence for sequence in sequences]

In [13]:
batch_size = 256
train_data = list(zip(train_sequences, train_targets))
valid_data = list(zip(valid_sequences, valid_targets))
test_data = list(zip(test_sequences, test_targets))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate)
valid_loader = DataLoader(valid_data, batch_size=batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, collate_fn=collate)

# RNN section

### RNN with nn.RNN
Implement a basic RNN model using PyTorch's built-in nn.RNN.
Define layers: embedding, RNN, and fully connected.

In [14]:
class RNNClassifier(nn.Module):
    def __init__(self, output_size, hidden_size, vocab_size,
                 device, n_layers=1,
                 embedding_dimension=50):
        super(RNNClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.device = device
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dimension, padding_idx=token2idx['<PAD>'])

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """Define Needed Layers"""
        self.rnn = nn.RNN(
            input_size=embedding_dimension,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, output_size)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """
        Implements the forward pass: first, embed the input tokens, then pass
        the embeddings through the RNN layer to capture sequential dependencies.
        Finally, use fully connected layers to output class probabilities.
        """
        embedded = self.embedding(inputs)

        rnn_output, hidden = self.rnn(embedded)

        last_hidden = hidden[-1]

        out = self.fc(last_hidden)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return out #probabilities for each class in the output.

### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

**Note**: You are not allowed to use library-built trainer functions in this section; the training loop should be implemented manually.

**Note**: To implement the training loop, you have the option to create a single train_model function that trains a model over multiple epochs, calculates training and validation accuracy, and logs the losses. Once written, this function can be reused for all RNN and LSTM models, allowing you to simply call it with different model instances for training. Reusing the function in this way will ensure that you receive credit for the training section of each subsequent model without needing to write separate loops , with just the correct function call.








In [16]:
model = RNNClassifier(
    output_size=2,
    hidden_size=128,
    vocab_size=len(token2idx),
    device=device,
    n_layers=1,
    embedding_dimension=50
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
def train_model(model, train_loader, valid_loader, criterion, optimizer, device, num_epochs=5):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "valid_loss": [],
        "valid_accuracy": []
    }

    for epoch in range(num_epochs):
        model.train()

        total_train_loss = 0.0
        train_correct = 0
        train_total = 0

        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} - Training"):
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * labels.size(0)

            predictions = torch.argmax(outputs, dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += labels.size(0)

        average_train_loss = total_train_loss / train_total
        train_accuracy = train_correct / train_total

        model.eval()

        total_valid_loss = 0.0
        valid_correct = 0
        valid_total = 0

        with torch.no_grad():
            for inputs, labels in tqdm(valid_loader, desc=f"Epoch {epoch + 1}/{num_epochs} - Validation"):
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                total_valid_loss += loss.item() * labels.size(0)

                predictions = torch.argmax(outputs, dim=1)
                valid_correct += (predictions == labels).sum().item()
                valid_total += labels.size(0)

        average_valid_loss = total_valid_loss / valid_total
        valid_accuracy = valid_correct / valid_total

        history["train_loss"].append(average_train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["valid_loss"].append(average_valid_loss)
        history["valid_accuracy"].append(valid_accuracy)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {average_train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Valid Loss: {average_valid_loss:.4f} | "
            f"Valid Acc: {valid_accuracy:.4f}"
        )

    return history


history_rnn = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=5
)

Epoch 1/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 18.71it/s]


Epoch 1/5 | Train Loss: 0.6835 | Train Acc: 0.5614 | Valid Loss: 0.9377 | Valid Acc: 0.0680


Epoch 2/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 22.18it/s]


Epoch 2/5 | Train Loss: 0.6743 | Train Acc: 0.5754 | Valid Loss: 0.7891 | Valid Acc: 0.1677


Epoch 3/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 14.64it/s]


Epoch 3/5 | Train Loss: 0.6696 | Train Acc: 0.5831 | Valid Loss: 0.9356 | Valid Acc: 0.1913


Epoch 4/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 10.19it/s]


Epoch 4/5 | Train Loss: 0.6522 | Train Acc: 0.6114 | Valid Loss: 0.8052 | Valid Acc: 0.4754


Epoch 5/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 20.80it/s]

Epoch 5/5 | Train Loss: 0.6605 | Train Acc: 0.5985 | Valid Loss: 0.7570 | Valid Acc: 0.2289


### RNN from Scratch
Implement an RNN from scratch by creating a custom RNN cell and a model that stacks these cells over time.

In [18]:
class CustomRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomRNNCell, self).__init__()
        self.hidden_size = hidden_size
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """Define Input-to-Hidden and Hidden-to-Hidden Layers"""
        self.input_to_hidden = nn.Linear(input_size, hidden_size)
        self.hidden_to_hidden = nn.Linear(hidden_size, hidden_size)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, input, hidden):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """
        Implements the forward pass: combines the input and previous hidden state
        to calculate the new hidden state for this RNN cell.
        """
        hidden = torch.tanh(
            self.input_to_hidden(input) + self.hidden_to_hidden(hidden)
        )

        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return hidden

In [19]:
class CustomRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(CustomRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=token2idx['<PAD>'])
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """Define Custom RNN Cell and Fully Connected Layers"""
        self.rnn_cell = CustomRNNCell(embedding_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################


    def forward(self, inputs):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """
        Implements the forward pass: performs embedding lookup, iterates through each
        time step, and passes embeddings through the custom RNN cell. Finally,
        applies the fully connected layers to output class probabilities.
        """
        embedded = self.embedding(inputs)

        batch_size = embedded.size(0)
        sequence_length = embedded.size(1)

        hidden = torch.zeros(batch_size, self.hidden_size).to(inputs.device)

        for time_step in range(sequence_length):
            current_input = embedded[:, time_step, :]
            hidden = self.rnn_cell(current_input, hidden)

        out = self.fc(hidden)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################
        return out #probabilities for each class in the output.

### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

In [21]:
model = CustomRNN(
    vocab_size=len(token2idx),
    embedding_dim=50,
    hidden_size=128,
    output_size=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:
############################# TODO #############################
# TODO: Implement the training loop
################################################################
history_custom_rnn = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=5
)










Epoch 1/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 14.81it/s]


Epoch 1/5 | Train Loss: 0.6526 | Train Acc: 0.6057 | Valid Loss: 0.6778 | Valid Acc: 0.6427


Epoch 2/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 19.43it/s]


Epoch 2/5 | Train Loss: 0.6501 | Train Acc: 0.6092 | Valid Loss: 0.7649 | Valid Acc: 0.4810


Epoch 3/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 19.21it/s]


Epoch 3/5 | Train Loss: 0.6538 | Train Acc: 0.6109 | Valid Loss: 0.7962 | Valid Acc: 0.3401


Epoch 4/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 19.85it/s]


Epoch 4/5 | Train Loss: 0.6399 | Train Acc: 0.6315 | Valid Loss: 0.7744 | Valid Acc: 0.3970


Epoch 5/5 - Validation: 100%|██████████| 10/10 [00:00<00:00, 20.46it/s]

Epoch 5/5 | Train Loss: 0.6411 | Train Acc: 0.6339 | Valid Loss: 0.7763 | Valid Acc: 0.4418


### evaluate RNN models on test set
To complete evaluate_on_test, loop through the test data to get predictions, calculate accuracy, and print a classification report for model evaluation. This function can be used to evaluate the performance LSTM models too.

**NOTE : to earn full marks for this section, you must adjust the network's hyperparameters so that each rnn models achieves at least 70% accuracy on the test data. If you achieve less than the required accuracy, consider adjusting your training loop and hyperparameters, such as the hidden state size and learning rate, to improve model performance.**

In [28]:
# Evaluate on test data
def evaluate_on_test(model, test_loader):
    model.eval()
    ############################# TODO #############################
    # TODO: Iterate over the test_loader, obtain model predictions,
    # calculate accuracy, and generate a classification report.
    ################################################################

    y_true_test = []
    y_pred_test = []

    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing"):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=1)

            test_correct += (predictions == labels).sum().item()
            test_total += labels.size(0)

            y_true_test.extend(labels.cpu().numpy())
            y_pred_test.extend(predictions.cpu().numpy())

    test_accuracy = test_correct / test_total
    print(f"Test Accuracy: {test_accuracy:.4f}")

    print(classification_report(y_true_test, y_pred_test))

In [29]:
# Evaluate both RNN models on the test dataset

print("Built-in nn.RNN model:")
evaluate_on_test(builtin_rnn_model, test_loader)

print("Custom RNN model:")
evaluate_on_test(custom_rnn_model, test_loader)

Built-in nn.RNN model:


Testing: 100%|██████████| 10/10 [00:00<00:00, 13.48it/s]
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]

Test Accuracy: 0.1461
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.15      0.25      2499

    accuracy                           0.15      2499
   macro avg       0.50      0.07      0.13      2499
weighted avg       1.00      0.15      0.25      2499

Custom RNN model:


Testing: 100%|██████████| 10/10 [00:00<00:00, 11.87it/s]

Test Accuracy: 0.1461
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.15      0.25      2499

    accuracy                           0.15      2499
   macro avg       0.50      0.07      0.13      2499
weighted avg       1.00      0.15      0.25      2499




~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# LSTM section

### LSTM with nn.LSTM
Define an LSTM model using PyTorch's built-in nn.LSTM.

In [30]:
class LSTMClassifier(nn.Module):
    def __init__(self, output_size, hidden_size, vocab_size,
                 device, bidirectional=False, n_layers=1,
                 embedding_dimension=50):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.device = device

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dimension, padding_idx = token2idx['<PAD>'])

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """Define the LSTM layer and fully connected layers"""
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=embedding_dimension,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional
        )

        self.fc = nn.Linear(hidden_size * self.num_directions, output_size)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        # Initialize hidden state and cell state with zeros
        hidden = torch.zeros(self.n_layers, inputs.size(0), self.hidden_size).to(inputs.device)
        cell_state = torch.zeros(self.n_layers, inputs.size(0), self.hidden_size).to(inputs.device)

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """
        Implements the forward pass: first, embed the input tokens, then pass
        the embeddings through the LSTM layer to capture sequential dependencies.
        Finally, use fully connected layers to output class probabilities.
        """
        embedded = self.embedding(inputs)

        hidden = torch.zeros(
            self.n_layers * self.num_directions,
            inputs.size(0),
            self.hidden_size
        ).to(inputs.device)

        cell_state = torch.zeros(
            self.n_layers * self.num_directions,
            inputs.size(0),
            self.hidden_size
        ).to(inputs.device)

        lstm_output, (hidden, cell_state) = self.lstm(embedded, (hidden, cell_state))

        if self.bidirectional:
            last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            last_hidden = hidden[-1]

        out = self.fc(last_hidden)
        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return out  # probabilities for each class in the output.

### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

**Note**: You are not allowed to use library-built trainer functions in this section; the training loop should be implemented manually.

**Note**: To implement the training loop, you have the option to create a single train_model function that trains a model over multiple epochs, calculates training and validation accuracy, and logs the losses. Once written, this function can be reused for all RNN and LSTM models, allowing you to simply call it with different model instances for training. Reusing the function in this way will ensure that you receive credit for the training section of each subsequent model without needing to write separate loops , with just the correct function call.

In [31]:
model = LSTMClassifier(
    output_size=2,
    hidden_size=128,
    vocab_size=len(token2idx),
    device=device,
    bidirectional=False,
    n_layers=1,
    embedding_dimension=50
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [32]:
############################# TODO #############################
# TODO: Implement the training loop
################################################################
history_lstm = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=5
)

lstm_model = model










Epoch 1/5 - Validation: 100%|██████████| 10/10 [00:01<00:00,  8.77it/s]


Epoch 1/5 | Train Loss: 0.6762 | Train Acc: 0.5719 | Valid Loss: 0.6321 | Valid Acc: 0.7919


Epoch 2/5 - Validation: 100%|██████████| 10/10 [00:01<00:00,  9.01it/s]


Epoch 2/5 | Train Loss: 0.5283 | Train Acc: 0.7422 | Valid Loss: 0.6463 | Valid Acc: 0.6715


Epoch 3/5 - Validation: 100%|██████████| 10/10 [00:01<00:00,  9.26it/s]


Epoch 3/5 | Train Loss: 0.4024 | Train Acc: 0.8230 | Valid Loss: 0.5272 | Valid Acc: 0.7947


Epoch 4/5 - Validation: 100%|██████████| 10/10 [00:01<00:00,  8.73it/s]


Epoch 4/5 | Train Loss: 0.3532 | Train Acc: 0.8501 | Valid Loss: 0.3200 | Valid Acc: 0.8591


Epoch 5/5 - Validation: 100%|██████████| 10/10 [00:01<00:00,  9.41it/s]

Epoch 5/5 | Train Loss: 0.3140 | Train Acc: 0.8686 | Valid Loss: 0.3512 | Valid Acc: 0.8515


### Custom LSTM from Scratch
Implement an LSTM from scratch by defining a LSTM cell and a model that combines these cells over the sequence.

In [33]:
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomLSTMCell, self).__init__()
        self.hidden_size = hidden_size
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """Define Needed Layers"""
        self.input_to_gates = nn.Linear(input_size, 4 * hidden_size)
        self.hidden_to_gates = nn.Linear(hidden_size, 4 * hidden_size)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, input, hidden, cell_state):
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """Define Forward pass"""
        gates = self.input_to_gates(input) + self.hidden_to_gates(hidden)

        input_gate, forget_gate, candidate_gate, output_gate = gates.chunk(4, dim=1)

        input_gate = torch.sigmoid(input_gate)
        forget_gate = torch.sigmoid(forget_gate)
        candidate_gate = torch.tanh(candidate_gate)
        output_gate = torch.sigmoid(output_gate)

        cell_state = forget_gate * cell_state + input_gate * candidate_gate
        hidden = output_gate * torch.tanh(cell_state)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return hidden, cell_state # New hidden state , New cell state

In [34]:
class CustomLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(CustomLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=token2idx['<PAD>'])
        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """Define Needed Layers"""
        self.lstm_cell = CustomLSTMCell(embedding_dim, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

    def forward(self, inputs):
        # Initialize hidden state and cell state with zeros
        hidden = torch.zeros(inputs.size(0), self.hidden_size).to(inputs.device)
        cell_state = torch.zeros(inputs.size(0), self.hidden_size).to(inputs.device)

        ###################################### TODO #####################################
        #                          COMPLETE THE FOLLOWING SECTION                       #
        #################################################################################
        """
        Implements the forward pass: first, embed the input tokens, then pass
        the embeddings through the LSTM layer to capture sequential dependencies.
        Finally, use fully connected layers to output class probabilities.
        """
        embedded = self.embedding(inputs)

        sequence_length = embedded.size(1)

        for time_step in range(sequence_length):
            current_input = embedded[:, time_step, :]
            hidden, cell_state = self.lstm_cell(current_input, hidden, cell_state)

        out = self.fc(hidden)

        #################################################################################
        #                                   THE END                                     #
        #################################################################################

        return out  # probabilities for each class in the output.

### Train model

In this section, you should train model for multiple epochs on the training data and evaluate it on the validation data after each epoch, reporting the model's accuracy. Ensure that the model is set to train mode during training and switched to eval mode for evaluation on the validation data. The objective is to implement the training loop and, at the next , compute and report the final accuracy on the test data.

In [35]:
model = CustomLSTM(
    vocab_size=len(token2idx),
    embedding_dim=50,
    hidden_size=128,
    output_size=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [36]:
############################# TODO #############################
# TODO: Implement the training loop
################################################################


history_custom_lstm = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=5
)

custom_lstm_model = model








Epoch 1/5 - Validation: 100%|██████████| 10/10 [00:02<00:00,  3.91it/s]


Epoch 1/5 | Train Loss: 0.6700 | Train Acc: 0.5846 | Valid Loss: 0.8157 | Valid Acc: 0.0700


Epoch 2/5 - Validation: 100%|██████████| 10/10 [00:02<00:00,  4.85it/s]


Epoch 2/5 | Train Loss: 0.6623 | Train Acc: 0.5940 | Valid Loss: 0.7253 | Valid Acc: 0.6679


Epoch 3/5 - Validation: 100%|██████████| 10/10 [00:02<00:00,  4.79it/s]


Epoch 3/5 | Train Loss: 0.6480 | Train Acc: 0.6148 | Valid Loss: 0.7849 | Valid Acc: 0.6347


Epoch 4/5 - Validation: 100%|██████████| 10/10 [00:02<00:00,  4.92it/s]


Epoch 4/5 | Train Loss: 0.4631 | Train Acc: 0.7841 | Valid Loss: 0.3421 | Valid Acc: 0.8571


Epoch 5/5 - Validation: 100%|██████████| 10/10 [00:02<00:00,  4.84it/s]

Epoch 5/5 | Train Loss: 0.3642 | Train Acc: 0.8388 | Valid Loss: 0.4585 | Valid Acc: 0.7915


### evaluate LSTM models on test set
To complete evaluate_on_test, loop through the test data to get predictions, calculate accuracy, and print a classification report for model evaluation.
you can use the `evaluate_on_test` function implemented in the previous section. Alternatively, you may write a new function to conduct this evaluation.ensure report the "classification_report" of both LSTM models.

**NOTE : to earn full marks for this section, you must adjust the network's hyperparameters so that each lstm models achieves at least 80% accuracy on the test data. If you achieve less than the required accuracy, consider adjusting your training loop and hyperparameters, such as the hidden state size and learning rate, to improve model performance.**

In [37]:
# Evaluate both LSTM models on the test dataset
print("Built-in LSTM model:")
evaluate_on_test(lstm_model, test_loader)

print("Custom LSTM model:")
evaluate_on_test(custom_lstm_model, test_loader)


Built-in LSTM model:


Testing: 100%|██████████| 10/10 [00:01<00:00,  5.45it/s]
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]

Test Accuracy: 0.8571
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.86      0.92      2499

    accuracy                           0.86      2499
   macro avg       0.50      0.43      0.46      2499
weighted avg       1.00      0.86      0.92      2499

Custom LSTM model:


Testing: 100%|██████████| 10/10 [00:02<00:00,  4.40it/s]

Test Accuracy: 0.7895
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.79      0.88      2499

    accuracy                           0.79      2499
   macro avg       0.50      0.39      0.44      2499
weighted avg       1.00      0.79      0.88      2499




~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
~\anaconda3\envs\dl_hw1\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# Testing RNN and LSTM Models on a New Review

In [38]:
# Example review
review = "It is no wonder that the film has such a high rating, it is quite literally breathtaking. What can I say that hasn't said before? Not much, it's the story, the acting, the premise, but most of all, this movie is about how it makes you feel. Sometimes you watch a film, and can't remember it days later, this film loves with you, once you've seen it, you don't forget."


## Preprocess the test Review
To prepare the review for the model, we need to follow similar preprocessing steps as we did for the dataset:

Remove special characters and convert the text to lowercase.
Tokenize the text into individual words.
Remove stopwords to focus only on meaningful words.
Convert tokens to indices based on the token2idx dictionary created earlier.
Pad or truncate the sequence to a length of max_len .


In [39]:
import torch
import re
from nltk import wordpunct_tokenize
from nltk.corpus import stopwords


# Preprocessing function
def preprocess_text(text, stop_words, token2idx, max_len):

    ########################### TODO ###########################
    # Step 1: Clean and lowercase the input text
    ################################################################

    text = re.sub(r"[^a-zA-Z\s]", " ", text).lower()

    ########################### TODO ###########################
    # Step 2: Tokenize the text into words
    # - Use wordpunct_tokenize to split the cleaned text into individual word tokens.
    ############################################################

    tokens = wordpunct_tokenize(text)

    ########################### TODO ###########################
    # Step 3: Remove stopwords from the token list
    ############################################################

    tokens = [token for token in tokens if token not in stop_words]

    ########################### TODO ###########################
    # Step 4: Convert tokens to indices based on the token2idx dictionary
    # - For each token in the list, get the corresponding index from the token2idx dictionary.
    # - If a token is not found in token2idx, replace it with the index of '<UNK>'.
    ################################################################

    tokens_idx = [token2idx.get(token, token2idx['<UNK>']) for token in tokens]

    ########################### TODO ###########################
    # Step 5: Pad or truncate the tokens_idx list to the desired max_len
    ############################################################

    if len(tokens_idx) < max_len:
        tokens_idx = tokens_idx + [token2idx['<PAD>']] * (max_len - len(tokens_idx))
    else:
        tokens_idx = tokens_idx[:max_len]

    ########################### End of TODOs ###########################

    return tokens_idx  # Return the processed list of indices

# Get stopwords
stop_words = set(stopwords.words('english'))

########################### TODO ###########################
# Preprocess the review
review_indices = preprocess_text(review, stop_words, token2idx, max_len)
############################################################

# Convert the indices to a tensor and move it to the device (GPU or CPU)
input_tensor = torch.LongTensor([review_indices]).to(device)

## Make Predictions
Now that we have preprocessed the review, use both the RNN and LSTM models to make predictions on the sentiment of the review.

Set the model to evaluation mode to prevent updates during inference.
Predict the sentiment class by passing the input_tensor to the model.
Interpret the prediction as either "Positive" or "Negative" based on the model's output.

In [40]:
def predict_sentiment(model, input_tensor, model_name="Model"):
    model.eval()  # Set the model to evaluation mode
    ############################# TODO #############################
    # TODO: Perform a forward pass with the model on the input_tensor,
    # get the predicted class label, and map it to "Positive" or "Negative".
    ################################################################

    with torch.no_grad():
        outputs = model(input_tensor)
        predicted_class = torch.argmax(outputs, dim=1).item()

    if predicted_class == 1:
        class_label = "Positive"
    else:
        class_label = "Negative"

    print(f"The predicted class for the review by {model_name} is: {class_label}")


 

In [41]:
# Make predictions using with "predict_sentiment" function for each of four models above

predict_sentiment(builtin_rnn_model, input_tensor, "Built-in RNN")
predict_sentiment(custom_rnn_model, input_tensor, "Custom RNN")
predict_sentiment(lstm_model, input_tensor, "Built-in LSTM")
predict_sentiment(custom_lstm_model, input_tensor, "Custom LSTM")



The predicted class for the review by Built-in RNN is: Negative
The predicted class for the review by Custom RNN is: Negative
The predicted class for the review by Built-in LSTM is: Positive
The predicted class for the review by Custom LSTM is: Positive


# Questions

[1] - Based on your observations, what do you think caused the difference in performance between the RNN and LSTM models (on test set)? Analyze this difference using the results from the notebook, and discuss where a simple RNN might perform better.

[2] - If we increase max_len in the preprocessing step to 300, what changes in models (rnn & lstm ) performance would you expect, and why? Please *explain* and discuss the impact this may have on the learning process and the final results.

<div dir="rtl" align="right">

## پاسخ سوال ۱

با توجه به نتیجه‌هایی که از نوت‌بوک گرفتم، عملکرد مدل‌ها خیلی با هم فرق داشته است. دقت مدل‌های RNN ساده تقریباً برابر با **0.1461** شده، هم برای مدل آماده‌ی `nn.RNN` و هم برای مدل `Custom RNN`. اما مدل LSTM آماده‌ی PyTorch دقت حدود **0.8571** گرفته و مدل `Custom LSTM` هم حدود **0.7895** گرفته است. یعنی در این اجرا، LSTMها خیلی بهتر از RNNها عمل کرده‌اند.

به نظر من دلیل اصلی این اختلاف این است که داده‌ی IMDB شامل reviewهای متنی نسبتاً طولانی است و برای تشخیص احساس متن، مدل باید بتواند اطلاعات بخش‌های مختلف جمله یا review را نگه دارد. مدل RNN ساده فقط یک hidden state دارد و در هر مرحله همان را به مرحله‌ی بعد منتقل می‌کند. به همین خاطر وقتی متن طولانی‌تر می‌شود، اطلاعات قسمت‌های ابتدایی متن کم‌کم ضعیف می‌شود و مدل نمی‌تواند وابستگی‌های بلندمدت را خوب یاد بگیرد. این همان مشکل معروف **vanishing gradient** است.

در نتیجه، RNNهای ساده در این تمرین احتمالاً نتوانسته‌اند الگوی واقعی متن را خوب یاد بگیرند. دقت حدود ۱۴٪ نشان می‌دهد که مدل عملاً رفتار پایداری نداشته و روی test set خوب generalize نکرده است. از طرف دیگر، LSTM چون دارای **cell state** و gateهای مختلف مثل forget gate، input gate و output gate است، بهتر می‌تواند تصمیم بگیرد چه اطلاعاتی را نگه دارد و چه اطلاعاتی را فراموش کند. همین باعث شده مدل LSTM آماده به دقت حدود ۸۵٪ برسد، که نسبت به RNN اختلاف خیلی زیادی دارد.

یک نکته‌ی مهم هم در خروجی‌ها دیده می‌شود: در `classification_report`، برای کلاس `0` مقدار `support` برابر صفر است. یعنی در این test set که گزارش شده، نمونه‌ای از کلاس منفی وجود نداشته یا test loader فقط نمونه‌های یک کلاس را وارد evaluation کرده است. بنابراین این گزارش از نظر ارزیابی دوکلاسه کامل نیست. با این حال، چون همین test set برای همه‌ی مدل‌ها استفاده شده، هنوز می‌توانیم مقایسه‌ی نسبی انجام بدهیم. در این مقایسه، LSTMها خیلی بهتر از RNNها بوده‌اند.

در مورد اینکه RNN ساده کجا ممکن است بهتر عمل کند، به نظرم در متن‌های کوتاه‌تر یا مسئله‌هایی که وابستگی بلندمدت زیادی ندارند، RNN می‌تواند مناسب باشد. مثلاً اگر جمله‌ها کوتاه باشند و کلمات مهم نزدیک به هم قرار گرفته باشند، RNN ساده هم می‌تواند الگوها را یاد بگیرد. همچنین RNN پارامترهای کمتری نسبت به LSTM دارد، ساده‌تر است، سریع‌تر train می‌شود و در بعضی داده‌های کوچک ممکن است کمتر overfit کند. بنابراین اگر مسئله ساده‌تر باشد یا طول sequence کم باشد، استفاده از RNN می‌تواند منطقی‌تر و سبک‌تر باشد.

اما در این تمرین خاص، چون داده‌ی IMDB متنی و نسبتاً طولانی است، نتیجه‌ها نشان می‌دهند که LSTM انتخاب مناسب‌تری بوده است.

---

## پاسخ سوال ۲

اگر مقدار `max_len` را در preprocessing به `300` افزایش بدهیم، یعنی به مدل اجازه می‌دهیم تعداد بیشتری از کلمات هر review را ببیند. در حالت فعلی، اگر طول متن از `max_len` بیشتر باشد، بخش آخر یا بخشی از review حذف می‌شود. بنابراین با افزایش `max_len`، اطلاعات بیشتری از متن اصلی وارد مدل می‌شود و از این نظر ممکن است عملکرد بهتر شود.

اما این تغییر برای RNN و LSTM اثر یکسانی ندارد.

برای مدل‌های **RNN ساده**، افزایش `max_len` احتمالاً باعث سخت‌تر شدن یادگیری می‌شود. چون RNN باید اطلاعات را در طول تعداد time step بیشتری منتقل کند. وقتی طول sequence به 300 برسد، مدل باید وابستگی‌های طولانی‌تری را یاد بگیرد، اما RNN ساده معمولاً در نگه داشتن اطلاعات بلندمدت ضعیف است. بنابراین ممکن است مشکل vanishing gradient شدیدتر شود و مدل حتی بیشتر گیج شود. با توجه به نتیجه‌ی فعلی RNNها که دقتشان فقط حدود **14.61%** شده، انتظار ندارم صرفاً افزایش `max_len` باعث بهتر شدن جدی آن‌ها شود. حتی ممکن است training کندتر شود یا performance بدتر هم بشود، چون مدل باید متن بلندتری را پردازش کند ولی ابزار مناسبی برای کنترل حافظه‌ی بلندمدت ندارد.

برای مدل‌های **LSTM**، افزایش `max_len` می‌تواند مفیدتر باشد. چون LSTM برای sequenceهای طولانی‌تر طراحی شده و با کمک cell state و gateها بهتر می‌تواند اطلاعات مهم را نگه دارد. در نتیجه، اگر بخش‌های مهم review بعد از طول فعلی بریده شده باشند، با `max_len=300` مدل LSTM می‌تواند آن بخش‌ها را هم ببیند و احتمالاً تصمیم دقیق‌تری بگیرد. مثلاً ممکن است review اول با چند جمله‌ی معمولی شروع شود، اما نظر اصلی کاربر در ادامه‌ی متن باشد. اگر آن بخش حذف شود، مدل اطلاعات ناقص می‌گیرد.

با این حال، افزایش `max_len` همیشه به معنی بهتر شدن accuracy نیست. چون sequence طولانی‌تر باعث چند مشکل هم می‌شود. اول اینکه زمان train بیشتر می‌شود، چون مدل باید تعداد time step بیشتری را پردازش کند. دوم اینکه مصرف حافظه بیشتر می‌شود. سوم اینکه اگر خیلی از reviewها کوتاه باشند، مقدار padding بیشتر می‌شود و مدل تعداد زیادی token بی‌معنی مثل `<PAD>` می‌بیند. همچنین متن بلندتر ممکن است noise بیشتری هم داشته باشد و همه‌ی کلمات اضافه واقعاً برای sentiment مفید نباشند.

با توجه به نتیجه‌های خودم، انتظار دارم اگر `max_len` به 300 افزایش پیدا کند، مدل `Built-in LSTM` که الان حدود **85.71%** دقت گرفته، احتمالاً بتواند همچنان عملکرد خوبی داشته باشد و شاید کمی بهتر شود، البته اگر overfitting یا noise زیاد نشود. مدل `Custom LSTM` هم که حدود **78.95%** گرفته، ممکن است با sequence بلندتر و تنظیم بهتر hyperparameterها به بالای 80٪ برسد، ولی چون پیاده‌سازی دستی آن ساده‌تر است، احتمالاً کندتر و حساس‌تر از `nn.LSTM` آماده خواهد بود.

در مقابل، برای RNNهای ساده انتظار زیادی برای بهبود ندارم. چون مشکل اصلی آن‌ها کم بودن اطلاعات ورودی نیست، بلکه ضعف در یادگیری وابستگی‌های بلندمدت است. وقتی `max_len` را بیشتر کنیم، دقیقاً همان بخشی از مسئله را سخت‌تر می‌کنیم که RNN در آن ضعیف است.

پس جمع‌بندی من این است که افزایش `max_len` به 300 احتمالاً برای LSTM مفیدتر از RNN است. LSTM ممکن است از اطلاعات بیشتر استفاده کند و نتیجه‌ی نهایی بهتری بدهد، اما RNN احتمالاً با sequence طولانی‌تر سخت‌تر train می‌شود و ممکن است دقت آن خیلی بهتر نشود. در کل، این تغییر هم‌زمان دو اثر دارد: از یک طرف اطلاعات بیشتری به مدل می‌دهد، ولی از طرف دیگر یادگیری را سنگین‌تر، کندتر و پرریسک‌تر می‌کند.

</div>
